# SerialEM_Locations_vs_CTF

This notebook requires access to:
1. At least one set of micrographs collected with SerialEM (with associated .mdoc files) 
2. Micrograph parameters .csv file downloaded from CryoSPARC. 

To obtain (2), build a Curate Exposures job, and then click "Download table CSV" (at the top of the micrographs table, next to the Filter dropdown)

The outputs of this job are images with one dot per micrograph, arrayed on a 2D grid resembling the atlas and colored by CTF Fit

## Imports and User Paths

Input the folder path that contains the .csv file(s) for the data set(s) you wish to visualize. This folder must not contain other unrelated .csv files. The file name before .csv will be the handle for your data set for the rest of the notebook, so you may want to name the file accordingly (i.e. tundra.csv, notilt.csv).

Additionally input path(s) to each set of .mdoc files. The script assumes the .mdoc files are in the same place as the original micrographs. 

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import re
import numpy as np
import seaborn as sns
from pathlib import Path
from functools import lru_cache


# Folder containing your CSV files
folder = "/path/to/directory"

mdoc_base_dirs = [
    Path("/path/to/first/mdoc/set"),
    Path("/path/to/second/mdoc/set"),
    Path("/path/to/third/mdoc/set"),
    Path("/path/to/fourth/mdoc/set")
]

In [ ]:
csv_files = sorted(glob.glob(os.path.join(folder, "*.csv")))

dfs = {}

for i, fpath in enumerate(csv_files, start=1):
    name = os.path.splitext(os.path.basename(fpath))[0]
    df = pd.read_csv(fpath)

    if "Index" in df.columns:
        df["Index"] = pd.to_numeric(df["Index"], errors="coerce")

    dfs[name] = df

    globals()[f"dataset{i}"] = name

In [ ]:
df_all = pd.concat(
    [df.copy().assign(dataset=name) for name, df in dfs.items()],
    ignore_index=True
)

In [ ]:
df_all

## Map the locations of each micrograph onto an LMM-level grid and combine with CTF information

Note that this segment assumes your movies are .eer files. If your movies are .tiffs or another format, you will have to change .eer to .tiff accordingly. 

In [ ]:
# Build an index: mdoc filename (basename) -> full path
mdoc_index = {}
for base in mdoc_base_dirs:
    for p in base.rglob("*.mdoc"):
        # if duplicates exist, first one wins; change as needed
        mdoc_index.setdefault(p.name, p)

def file_to_mdoc_basename(file_value: str) -> str:
    """Convert the df 'File' value to the expected .mdoc basename: <something>.eer.mdoc"""
    s = Path(str(file_value)).name
    if "_patch_aligned_doseweighted.mrc" in s:
        s = s[: s.find("_patch_aligned_doseweighted.mrc")]  # keep through ".mrc"        
        return s + ".eer.mdoc"
    if s.endswith(".mdoc"):
        return s
    if s.endswith(".eer"):
        return s + ".mdoc"
    if ".eer" in s:
        s = s[: s.find(".eer") + 4]  # keep through ".eer"
        return s + ".mdoc"
    return s # fallback

df_all["mdoc_basename"] = df_all["File"].astype(str).map(file_to_mdoc_basename)
df_all["mdoc_path"] = df_all["mdoc_basename"].map(lambda n: mdoc_index.get(n, pd.NA))

@lru_cache(maxsize=None)
def parse_mdoc(path_str: str) -> dict:
    """
    Parse selected keys from the [FrameSet = 0] section of a SerialEM .mdoc file.
    Returns a dict with:
      TiltAngle, StagePositionX, StagePositionY, Defocus, ImageShiftX, ImageShiftY,
      RotationAngle, FlashCounter, FEGCurrent
    """
    out = {
        "TiltAngle": np.nan,
        "StagePositionX": np.nan, "StagePositionY": np.nan,
        "Defocus": np.nan,
        "ImageShiftX": np.nan, "ImageShiftY": np.nan,
        "RotationAngle": np.nan,
        "FlashCounter": np.nan,
        "FEGCurrent": np.nan,
    }

    p = Path(path_str)
    if not p.exists():
        return out

    wanted = {
        "TiltAngle", "StagePosition", "Defocus", "ImageShift",
        "RotationAngle", "FlashCounter", "FEGCurrent"
    }

    in_frameset0 = False
    with p.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            if line.startswith("[FrameSet"):
                # accept "[FrameSet = 0]" and minor spacing variants
                in_frameset0 = ("0" in line and "=" in line and line.replace(" ", "").startswith("[FrameSet=0]"))
                continue

            if not in_frameset0 or "=" not in line:
                continue

            key, val = [x.strip() for x in line.split("=", 1)]
            if key not in wanted:
                continue

            if key in ("TiltAngle", "Defocus", "RotationAngle", "FEGCurrent"):
                out[key] = pd.to_numeric(val, errors="coerce")

            elif key == "FlashCounter":
                out[key] = pd.to_numeric(val, errors="coerce")

            elif key == "StagePosition":
                parts = val.split()
                if len(parts) >= 2:
                    out["StagePositionX"] = pd.to_numeric(parts[0], errors="coerce")
                    out["StagePositionY"] = pd.to_numeric(parts[1], errors="coerce")

            elif key == "ImageShift":
                parts = val.split()
                if len(parts) >= 2:
                    out["ImageShiftX"] = pd.to_numeric(parts[0], errors="coerce")
                    out["ImageShiftY"] = pd.to_numeric(parts[1], errors="coerce")

    return out

meta = df_all["mdoc_path"].dropna().astype(str).map(parse_mdoc)
meta_df = pd.json_normalize(meta).reindex(df_all.index)  # align back to df_all rows

df_all = pd.concat([df_all, meta_df], axis=1)

In [ ]:
df_all.head(2)

## Visualize micrograph locations and color by CTF - No Beam Shift

The following cells currently run for one dataset. If you have two datasets, you will have to unhash the dataset2 information. You can add more datasets as desired. 

You may want to change (on CTF plots):
1. vmin/vmax (minimum and maximum CTF values for color scale)
2. xlim/ylim (axis bounds, particularly to zoom in on a grid square)
3. s (size of dots)

Remove the hash to save any of the figures once they look as desired.

In [ ]:
df_stats = df_all
x_axis = 'StagePositionX'
y_axis = 'StagePositionY'

plt.scatter(df_stats[df_stats['dataset'] == dataset1][x_axis], df_stats[df_stats['dataset'] == dataset1][y_axis], alpha=1, label=dataset1, color='blue', s=1)
#plt.scatter(df_stats[df_stats['dataset'] == dataset2][x_axis], df_stats[df_stats['dataset'] == dataset2][y_axis], alpha=1, label=dataset2, color='orange', s=1)

plt.legend(markerscale=5)
plt.xlabel('StagePositionX')
plt.ylabel('StagePositionY')
plt.grid(False)
plt.xlim(-999,999)
plt.ylim(-999,999)
# plt.savefig(f"{folder}/Square_locations.png")
plt.show()

In [ ]:
x_axis = 'StagePositionX'
y_axis = 'StagePositionY'

plt.scatter(df_stats[df_stats['dataset'] == dataset1][x_axis], df_stats[df_stats['dataset'] == dataset1][y_axis], alpha=1, label=dataset1, c=df_stats[df_stats['dataset'] == dataset1]["CTF Fit"], vmin=2, vmax=10, cmap='Blues_r', s=1)
#plt.scatter(df_stats[df_stats['dataset'] == dataset2][x_axis], df_stats[df_stats['dataset'] == dataset2][y_axis], alpha=1, label=dataset2, c=df_stats[df_stats['dataset'] == dataset2]["CTF Fit"], vmin=2, vmax=10, cmap='Greens_r', s=1)

plt.legend(markerscale=5)
cbar = plt.colorbar()
cbar.set_label("CTF Fit")
plt.xlabel('StagePositionX')
plt.ylabel('StagePositionY')
plt.grid(False)
plt.xlim(-999,999)
plt.ylim(-999,999)
#plt.savefig(f"{folder}/CTF_v_stage_position.png")
plt.show()

In [ ]:
x_axis = 'StagePositionX'
y_axis = 'StagePositionY'

plt.scatter(df_stats[df_stats['dataset'] == dataset1][x_axis], df_stats[df_stats['dataset'] == dataset1][y_axis], alpha=1, label=dataset1, c=df_stats[df_stats['dataset'] == dataset1]["CTF Fit"], vmin=2, vmax=10, cmap='Blues_r', s=1)
#plt.scatter(df_stats[df_stats['dataset'] == dataset2][x_axis], df_stats[df_stats['dataset'] == dataset2][y_axis], alpha=1, label=dataset2, c=df_stats[df_stats['dataset'] == dataset2]["CTF Fit"], vmin=2, vmax=10, cmap='Greens_r', s=1)

plt.legend(markerscale=5)
cbar = plt.colorbar()
cbar.set_label("CTF Fit")
plt.xlabel('StagePositionX')
plt.ylabel('StagePositionY')
plt.grid(False)
plt.xlim(-750,0)
plt.ylim(0,750)
#plt.savefig(f"{folder}/CTF_v_stage_position_zoom.png")
plt.show()

## Visualize micrograph locations and color by CTF - With Beam Shift

The following cells currently run for one dataset. If you have two datasets, you will have to unhash the dataset2 information. You can add more datasets as desired. 

You may want to change (on CTF plots):
1. vmin/vmax (minimum and maximum CTF values for color scale)
2. xlim/ylim (axis bounds, particularly to zoom in on a grid square)
3. s (size of dots)

Remove the hash to save any of the figures once they look as desired.

In [ ]:
df_stats["TrueXPosition"] = df_stats["StagePositionX"] + df_stats["ImageShiftX"]
df_stats["TrueYPosition"] = df_stats["StagePositionY"] + df_stats["ImageShiftY"]

In [ ]:
x_axis = 'TrueXPosition'
y_axis = 'TrueYPosition'

plt.scatter(df_stats[df_stats['dataset'] == dataset1][x_axis], df_stats[df_stats['dataset'] == dataset1][y_axis], alpha=1, label=dataset1, c=df_stats[df_stats['dataset'] == dataset1]["CTF Fit"], vmin=2, vmax=10, cmap='Blues_r', s=1)
#plt.scatter(df_stats[df_stats['dataset'] == dataset2][x_axis], df_stats[df_stats['dataset'] == dataset2][y_axis], alpha=1, label='No Tilt Bad', c=df_stats[df_stats['dataset'] == dataset2]["CTF Fit"], vmin=2, vmax=10, cmap='Greens_r', s=1)

plt.legend(markerscale=5)
cbar = plt.colorbar()
cbar.set_label("CTF Fit")
plt.xlabel('TruePositionX')
plt.ylabel('TruePositionY')
plt.grid(False)
plt.xlim(-999,999)
plt.ylim(-999,999)
#plt.savefig(f"{folder}/CTF_v_true_position.png")
plt.show()

## Coloring by Defocus or Ice Thickness

The following cells currently run for one dataset. If you have two datasets, you will have to unhash the dataset2 information. You can add more datasets as desired. 

You may want to change (on CTF plots):
1. vmin/vmax (minimum and maximum defocus or ice thickness values for color scale)
2. xlim/ylim (axis bounds, particularly to zoom in on a grid square)
3. s (size of dots)

Remove the hash to save any of the figures once they look as desired.

In [ ]:
x_axis = 'TrueXPosition'
y_axis = 'TrueYPosition'

plt.scatter(df_stats[df_stats['dataset'] == dataset1][x_axis], df_stats[df_stats['dataset'] == dataset1][y_axis], alpha=1, label=dataset1, c=df_stats[df_stats['dataset'] == dataset1]["Defocus"], vmin=-3, vmax=0, cmap='Blues', s=1)
cbar = plt.colorbar()
# plt.scatter(df_stats[df_stats['dataset'] == dataset2][x_axis], df_stats[df_stats['dataset'] == dataset2][y_axis], alpha=1, label=dataset2, c=df_stats[df_stats['dataset'] == dataset2]["Defocus"], vmin=-3, vmax=0, cmap='Greens', s=1)

plt.legend(markerscale=5)
cbar.set_label("Defocus (um)")
plt.xlabel('TruePositionX')
plt.ylabel('TruePositionY')
plt.grid(False)
plt.xlim(-999,999)
plt.ylim(-999,999)
plt.savefig(f"{folder}/Defocus_v_true_position.png")
plt.show()

In [ ]:
x_axis = 'TrueXPosition'
y_axis = 'TrueYPosition'

plt.scatter(df_stats[df_stats['dataset'] == dataset1][x_axis], df_stats[df_stats['dataset'] == dataset1][y_axis], alpha=1, label=dataset1, c=df_stats[df_stats['dataset'] == dataset1]["Rel Ice Thick."], vmin=1.1, vmax=1.15, cmap='Blues_r', s=1)
cbar = plt.colorbar()
#plt.scatter(df_stats[df_stats['dataset'] == dataset2][x_axis], df_stats[df_stats['dataset'] == dataset2][y_axis], alpha=1, label=dataset2, c=df_stats[df_stats['dataset'] == dataset2]["Rel Ice Thick."], vmin=1.1, vmax=1.15, cmap='Greens_r', s=1)

plt.legend(markerscale=5)
cbar.set_label("Relative Ice Thickness")
plt.xlabel('TruePositionX')
plt.ylabel('TruePositionY')
plt.grid(False)
plt.xlim(-999,999)
plt.ylim(-999,999)
#plt.savefig(f"{folder}/IceThickness_v_true_position.png")
plt.show()